In [1]:
from ipyleaflet import Map, CircleMarker, basemaps, Polyline
from _main import curvecreator
from mainhouse import curvecreatorh
#from import rowcreator
#from IPython.display import display, HTML
import buildcache
import geopandas as gpd
from shapely.geometry import Point

def jupmain():
    #initialize map
    m = Map(
        center=(46.4983, 11.3548),
        zoom=13,
        basemap=basemaps.OpenStreetMap.Mapnik,
        layout={'width': '100%', 'height': '1000px'},#'90vh'},
        scroll_wheel_zoom=True
    )
    
    #display building centroids as markers (unused in main veriosn)
    def display_building_points(map_obj, geodataframe):
    
        if geodataframe is None:
            return
    
        for geom in geodataframe.geometry:
            map_obj.add_layer(
                CircleMarker(
                    location=(geom.y, geom.x),
                    radius=2,
                    color="salmon"
                )
            )
            
    #this is the best coordinate system to use and is based on the location of the first click
    utm_crs = None

    #initialize polist
    polist = []  # (x, y) in lokalem UTM, Meter
    polist_wgs84 = []    # (lat, lon) in EPSG:4326
    #avoid redrawing buildings every cycle (unused in latesdt)
    buildings_displayed = False
    
    
    def handle_click(**kwargs):
        global buildings_displayed
    
        if kwargs.get("type") != "click":
            return
    
        #display(HTML(f"<p>Current points: {polist}</p>"))#this can go
    
        lat, lon = kwargs.get("coordinates")
        #polist.append(latlng)#stores as degrees
        polist_wgs84.append((lat, lon))
        
        point_wgs84 = gpd.GeoSeries(
            [Point(lon, lat)],#why tf is it lat lon then lon lat
            crs="EPSG:4326"
        )

        # Determine local UTM zone from first click
        if utm_crs is None:
            utm_crs = point_wgs84.estimate_utm_crs()

        point_utm = point_wgs84.to_crs(utm_crs).iloc[0]
        polist.append((point_utm.x, point_utm.y))#create local utm polist
        
        # Store as meters
        #polist.append(
        #    (point_utm.x, point_utm.y)
        #)

    
        #show clicked points
        m.add_layer(
            CircleMarker(
                location=latlng,
                radius=4
            )
        )
    
        #create line and cache on fourth click
        if len(polist) % 4 == 0:
    
            ultquatr = polist[-4:]
    
            #point2 = polist[-3]
            #point3 = polist[-2]
            point2_wgs84 = polist_wgs84[-3]#new version with diffrent polist name when using wgs84
            point3_wgs84 = polist_wgs84[-2]
            #builds cache once if uncommented
            #if buildcache.building_gdf is None:
    
            buildcache.building_cache(
                point2,
                point3,
                utm_crs
            )
    
            display_building_points( #uncomment only for very small amounts of houses, else takes extremly long
                m,
                buildcache.building_gdf_wgs84
            )
    
            buildings_displayed = True
    
            ###curve creation part###
    
            #angular curve
            #lineplot = curvecreator(ultquatr)
    
            #line = Polyline(
            #    locations=lineplot,
            #    color="blue",
            #    weight=3,
            #    fill=False
            #)
            #m.add_layer(line)
            
            #house curve
            lineploth = curvecreatorh(ultquatr, buildcache.building_gdf)#building gdf as variable doesnt work
            
            lineh = Polyline(
                locations=lineploth,
                color="red",
                weight=3,
                fill=False
            )
            m.add_layer(lineh)
    
            #combined curve
            #rowplot = rowcreator(ultraqatr, building_gdf)
            
            #row = Polyline(
            #    locations=rowplot,
            #    color="purple",
            #    weight=3,
            #    fill=False
            #)
            #m.add_layer(row)
    
    #attach click handler
    m.on_interaction(handle_click)
    
    #display map
    display(m)

jupmain()

Map(center=[46.4983, 11.3548], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…